# PINN vs FEM Comparison — Colab Notebook

Compare the trained PINN against David's FEM reference solution on the same 91×91 grid.

**Before running:** put `pinn_bio_mms_model.pt` and `Data.zip` in your Google Drive
under `MyDrive/PINN_FEM/` (or edit the paths in Cell 2).

Each cell is independent — run them in order from the top.


## Cell 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Cell 2 — Set paths and copy/unzip files into the Colab runtime


In [ ]:
# EDIT THESE TWO PATHS to match where you put the files on your Drive.
# Examples below assume you put both in MyDrive/PINN_FEM/.
DRIVE_MODEL_PATH = '/content/drive/MyDrive/PINN_FEM/pinn_bio_mms_model.pt'
DRIVE_DATA_ZIP   = '/content/drive/MyDrive/PINN_FEM/Data.zip'

# Working directory inside Colab
import os, shutil, zipfile
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

# Stage the model
PINN_DIR = os.path.join(WORK, 'pinn_bio_mms')
os.makedirs(PINN_DIR, exist_ok=True)
shutil.copy(DRIVE_MODEL_PATH, os.path.join(PINN_DIR, 'pinn_bio_mms_model.pt'))
print(f"Copied model -> {PINN_DIR}/pinn_bio_mms_model.pt")

# Unzip FEM data
shutil.copy(DRIVE_DATA_ZIP, os.path.join(WORK, 'Data.zip'))
with zipfile.ZipFile(os.path.join(WORK, 'Data.zip'), 'r') as zf:
    zf.extractall(WORK)
print(f"Unzipped Data -> {WORK}/Data/")

# Verify layout
print("\nDirectory tree:")
for d in sorted(os.listdir(WORK)):
    full = os.path.join(WORK, d)
    if os.path.isdir(full):
        sub = sorted(os.listdir(full))[:6]
        print(f"  {d}/  ->  {sub}{'...' if len(os.listdir(full)) > 6 else ''}")
    else:
        print(f"  {d}  ({os.path.getsize(full)} bytes)")


## Cell 3 — Imports + setup


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")


## Cell 4 — Define BioMMS and PINN_HardIC (must match training file)


In [ ]:
class BioMMS:
    def __init__(self, L=1.0):
        self.L = L
        self.k = 2.0 * np.pi**2 / L**2

    def phi_t(self, x, y):
        return torch.cos(np.pi * x / self.L) * torch.cos(np.pi * y / self.L)

    def phi_n(self, x, y):
        return np.cos(np.pi * x / self.L) * np.cos(np.pi * y / self.L)

    def exact_n(self, x, y, t):
        p = self.phi_n(x, y)
        T = (0.4 + 0.2 * p) * np.exp(-0.3 * t)
        B = (0.08 + 0.03 * p) * (1.0 - np.exp(-0.5 * t))
        O = 0.2 + 0.02 * p * np.exp(-t)
        I = (0.02 + 0.008 * p) * (1.0 - np.exp(-0.3 * t))
        S = (0.05 + 0.02 * p) * (1.0 - np.exp(-0.5 * t))
        return T, B, O, I, S

    def ic_t(self, x, y):
        p = self.phi_t(x, y)
        T0 = 0.4 + 0.2 * p
        B0 = torch.zeros_like(p)
        O0 = 0.2 + 0.02 * p
        I0 = torch.zeros_like(p)
        S0 = torch.zeros_like(p)
        return torch.cat([T0, B0, O0, I0, S0], dim=1)


class PINN_HardIC(nn.Module):
    def __init__(self, layers, mms):
        super().__init__()
        self.act = nn.Tanh()
        self.lins = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.lins.append(nn.Linear(layers[i], layers[i + 1]))
        self.mms = mms

    def forward(self, x, y, t):
        inp = torch.cat([x, y, t], dim=1)
        for l in self.lins[:-1]:
            inp = self.act(l(inp))
        return self.mms.ic_t(x, y) + t * self.lins[-1](inp)


## Cell 5 — Load the trained PINN


In [ ]:
PINN_MODEL_PATH = os.path.join(PINN_DIR, 'pinn_bio_mms_model.pt')
ckpt = torch.load(PINN_MODEL_PATH, map_location=device, weights_only=False)
LAYERS = ckpt['layers']
mms = BioMMS(L=1.0)
model = PINN_HardIC(LAYERS, mms).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded PINN: layers={LAYERS}, "
      f"params={sum(p.numel() for p in model.parameters())}, "
      f"train wall time={ckpt.get('total_time', 0):.0f}s")


def predict_pinn(x_flat, y_flat, t_val):
    """Evaluate PINN at 1D arrays of (x, y) for a single time t."""
    with torch.no_grad():
        xt = torch.as_tensor(x_flat, dtype=torch.float32,
                             device=device).unsqueeze(1)
        yt = torch.as_tensor(y_flat, dtype=torch.float32,
                             device=device).unsqueeze(1)
        tt = torch.full((len(x_flat), 1), float(t_val),
                        dtype=torch.float32, device=device)
        return model(xt, yt, tt).cpu().numpy()


## Cell 6 — Locate FEM directories, identify grid + times


In [ ]:
DATA_DIR = os.path.join(WORK, 'Data')
EX_DIR = os.path.join(DATA_DIR, 'Solution_exacte')

ap_candidates = [d for d in os.listdir(DATA_DIR)
                 if d.startswith('Solution_approch') and d != 'Solution_exacte']
assert ap_candidates, f"Could not find Solution_approch* in {DATA_DIR}"
AP_DIR = os.path.join(DATA_DIR, ap_candidates[0])
print(f"FEM exact  : {EX_DIR}")
print(f"FEM approx : {AP_DIR}")

# Time labels (sorted numerically)
tap_dir = os.path.join(AP_DIR, 'Tap')
time_entries = sorted([(float(f), f) for f in os.listdir(tap_dir)])
time_vals = np.array([t for t, _ in time_entries])
time_names = [n for _, n in time_entries]
print(f"FEM timesteps: {len(time_entries)} "
      f"(t in [{time_vals[0]:.2f}, {time_vals[-1]:.2f}])")

# Grid info from one file
sample = np.loadtxt(os.path.join(tap_dir, time_names[0]))
N_grid = sample.shape[0]
nx = int(np.sqrt(N_grid))
assert nx * nx == N_grid, f"FEM grid not square: {N_grid} points"
h = 1.0 / (nx - 1)
x_ref = sample[:, 0]
y_ref = sample[:, 1]
print(f"FEM grid: {nx}x{nx} = {N_grid} points (h = {h:.6f})")


## Cell 7 — Sanity check: David's exact == our analytical MMS


In [ ]:
print("Verifying FEM exact solution matches the analytical MMS...")
max_diff = 0.0
ex_subs = [('T','Tex',0),('B','Bex',1),('O','Oex',2),('I','Iex',3),('S','Sex',4)]
for t_val, t_name in time_entries[::20]:
    for vname, ex_sub, idx in ex_subs:
        d = np.loadtxt(os.path.join(EX_DIR, ex_sub, t_name))
        analytic = mms.exact_n(d[:, 0], d[:, 1], t_val)[idx]
        max_diff = max(max_diff, np.abs(d[:, 2] - analytic).max())
print(f"  max |FEM_exact - analytical| = {max_diff:.2e}  (should be << 1)")


## Cell 8 — Evaluate PINN on FEM grid + compute errors


In [ ]:
vn = ['T', 'B', 'O', 'I', 'S']
ap_sub_map = {'T':'Tap','B':'Bap','O':'Oap','I':'Iap','S':'Sap'}
ex_sub_map = {'T':'Tex','B':'Bex','O':'Oex','I':'Iex','S':'Sex'}

pinn_rel = {v: [] for v in vn}; pinn_abs = {v: [] for v in vn}
fem_rel  = {v: [] for v in vn}; fem_abs  = {v: [] for v in vn}

for k, (t_val, t_name) in enumerate(time_entries):
    ex_data = {}
    for v in vn:
        ex_data[v] = np.loadtxt(
            os.path.join(EX_DIR, ex_sub_map[v], t_name))[:, 2]
    pinn_pred = predict_pinn(x_ref, y_ref, t_val)
    fem_data = {}
    for v in vn:
        fem_data[v] = np.loadtxt(
            os.path.join(AP_DIR, ap_sub_map[v], t_name))[:, 2]

    for j, v in enumerate(vn):
        ue = ex_data[v]
        nrm = np.sqrt(h*h * np.sum(ue**2))
        L2_p = np.sqrt(h*h * np.sum((pinn_pred[:, j] - ue)**2))
        L2_f = np.sqrt(h*h * np.sum((fem_data[v]    - ue)**2))
        pinn_abs[v].append(L2_p)
        fem_abs[v].append(L2_f)
        pinn_rel[v].append(L2_p / nrm if nrm > 1e-10 else float('nan'))
        fem_rel[v].append(L2_f / nrm if nrm > 1e-10 else float('nan'))

    if (k + 1) % 25 == 0:
        print(f"  {k+1:>3}/{len(time_entries)} times processed")

print("Errors computed.")


## Cell 9 — Print summary table


In [ ]:
def maxat(arr, t, vname):
    a = np.array(arr)
    if vname in ('B', 'I', 'S'):
        m = t >= 0.1
        i = np.nanargmax(a[m]); return a[m][i], t[m][i]
    i = np.nanargmax(a); return a[i], t[i]

print(f"\n{'='*78}")
print(f"PINN vs FEM ERROR SUMMARY (max over t in [0,1], grid {nx}x{nx})")
print(f"{'='*78}")
print(f"{'Var':<5}{'PINN rel':>14}{'FEM rel':>14}{'ratio':>9}"
      f"{'PINN abs':>14}{'FEM abs':>14}{'ratio':>9}")
print('-' * 79)
for v in vn:
    pr, _ = maxat(pinn_rel[v], time_vals, v)
    fr, _ = maxat(fem_rel[v],  time_vals, v)
    pa, _ = maxat(pinn_abs[v], time_vals, v)
    fa, _ = maxat(fem_abs[v],  time_vals, v)
    print(f"{v:<5}{pr:>14.3e}{fr:>14.3e}{fr/pr:>8.2f}x"
          f"{pa:>14.3e}{fa:>14.3e}{fa/pa:>8.2f}x")
print(f"{'='*79}")
print("ratio > 1 means PINN more accurate (FEM error / PINN error).")
print("For B, I, S the relative L2 is reported from t >= 0.1 only.")


## Cell 10 — Figure 1: error vs time (2 rows, 5 cols)


In [ ]:
OUT_DIR = os.path.join(WORK, 'comparison_results')
os.makedirs(OUT_DIR, exist_ok=True)

vc_pinn = '#1565c0'  # blue
vc_fem  = '#d32f2f'  # red
vlabels = {'T':'$T$','B':'$B$','O':'$O$','I':'$I$','S':'$S$'}
ta = time_vals

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
fig.suptitle(f'PINN vs FEM Errors  -  Biological-Scale MMS  '
             f'(grid {nx}x{nx})', fontsize=15, weight='bold', y=1.00)

for j, v in enumerate(vn):
    # relative
    ax = axes[0, j]
    if v in ('B', 'I', 'S'):
        m = ta >= 0.1
        ax.semilogy(ta[m], np.array(pinn_rel[v])[m], '-',
                    color=vc_pinn, lw=2.0, label='PINN')
        ax.semilogy(ta[m], np.array(fem_rel[v])[m], '--',
                    color=vc_fem,  lw=2.0, label='FEM')
        ax.text(0.04, 0.04, '$t \\geq 0.1$', transform=ax.transAxes,
                fontsize=8, color='gray')
    else:
        ax.semilogy(ta, pinn_rel[v], '-',  color=vc_pinn, lw=2.0, label='PINN')
        ax.semilogy(ta, fem_rel[v],  '--', color=vc_fem,  lw=2.0, label='FEM')
    ax.set_xlabel('Time $t$')
    ax.set_ylabel('Relative $L^2$ error')
    ax.set_title(f'{vlabels[v]}  (relative)', fontsize=12, weight='bold')
    ax.grid(True, alpha=0.3, which='both')
    if j == 0: ax.legend(loc='best', fontsize=10)

    # absolute
    ax = axes[1, j]
    ax.semilogy(ta, pinn_abs[v], '-',  color=vc_pinn, lw=2.0, label='PINN')
    ax.semilogy(ta, fem_abs[v],  '--', color=vc_fem,  lw=2.0, label='FEM')
    ax.set_xlabel('Time $t$')
    ax.set_ylabel('Absolute $L^2$ error')
    ax.set_title(f'{vlabels[v]}  (absolute)', fontsize=12, weight='bold')
    ax.grid(True, alpha=0.3, which='both')
    if j == 0: ax.legend(loc='best', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'pinn_vs_fem_errors.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'pinn_vs_fem_errors.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## Cell 11 — Figure 2: pointwise error snapshot at t = 0.5


In [ ]:
t_snap = 0.5
i_snap = int(np.argmin(np.abs(ta - t_snap)))
t_snap_actual = ta[i_snap]
t_name_snap = time_names[i_snap]
print(f"Snapshot time: t = {t_snap_actual:.2f}")

X = x_ref.reshape(nx, nx); Y = y_ref.reshape(nx, nx)
ex_snap = {}
for v in vn:
    ex_snap[v] = np.loadtxt(
        os.path.join(EX_DIR, ex_sub_map[v], t_name_snap))[:, 2].reshape(nx, nx)
pinn_snap = predict_pinn(x_ref, y_ref, t_snap_actual)
fem_snap = {}
for v in vn:
    fem_snap[v] = np.loadtxt(
        os.path.join(AP_DIR, ap_sub_map[v], t_name_snap))[:, 2].reshape(nx, nx)

fig, axes = plt.subplots(2, 5, figsize=(22, 8.5))
fig.suptitle(f'Pointwise error  $|u - u_{{\\mathrm{{exact}}}}|$  '
             f'at $t = {t_snap_actual:.2f}$  (grid {nx}x{nx})',
             fontsize=15, weight='bold', y=1.00)

for j, v in enumerate(vn):
    ue = ex_snap[v]
    up = pinn_snap[:, j].reshape(nx, nx)
    uf = fem_snap[v]
    err_p = np.abs(up - ue)
    err_f = np.abs(uf - ue)
    err_hi = max(err_p.max(), err_f.max(), 1e-12)

    im0 = axes[0, j].pcolormesh(X, Y, err_p, cmap='inferno',
                                shading='auto', vmin=0, vmax=err_hi)
    plt.colorbar(im0, ax=axes[0, j], fraction=0.046, pad=0.04)
    axes[0, j].set_aspect('equal')
    axes[0, j].set_title(f'PINN  {vlabels[v]}', fontsize=12, weight='bold')
    axes[0, j].set_xticks([0, 0.5, 1]); axes[0, j].set_yticks([0, 0.5, 1])

    im1 = axes[1, j].pcolormesh(X, Y, err_f, cmap='inferno',
                                shading='auto', vmin=0, vmax=err_hi)
    plt.colorbar(im1, ax=axes[1, j], fraction=0.046, pad=0.04)
    axes[1, j].set_aspect('equal')
    axes[1, j].set_title(f'FEM  {vlabels[v]}', fontsize=12, weight='bold')
    axes[1, j].set_xticks([0, 0.5, 1]); axes[1, j].set_yticks([0, 0.5, 1])

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'snapshot_t0.5.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'snapshot_t0.5.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## Cell 12 — Figure 3: summary bar chart


In [ ]:
def maxv_arr(d, vlist, t, _filter=True):
    out = []
    for v in vlist:
        a = np.array(d[v])
        if _filter and v in ('B', 'I', 'S'):
            out.append(np.nanmax(a[t >= 0.1]))
        else:
            out.append(np.nanmax(a))
    return np.array(out)

pinn_relmax = maxv_arr(pinn_rel, vn, ta, True)
fem_relmax  = maxv_arr(fem_rel,  vn, ta, True)
pinn_absmax = maxv_arr(pinn_abs, vn, ta, False)
fem_absmax  = maxv_arr(fem_abs,  vn, ta, False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xpos = np.arange(len(vn)); width = 0.35

axes[0].bar(xpos - width/2, pinn_relmax, width, color=vc_pinn, label='PINN')
axes[0].bar(xpos + width/2, fem_relmax,  width, color=vc_fem,  label='FEM')
axes[0].set_yscale('log')
axes[0].set_xticks(xpos); axes[0].set_xticklabels([vlabels[v] for v in vn])
axes[0].set_ylabel('Max relative $L^2$ error')
axes[0].set_title('Relative $L^2$ (max over $t$)', fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='y', which='both'); axes[0].legend()

axes[1].bar(xpos - width/2, pinn_absmax, width, color=vc_pinn, label='PINN')
axes[1].bar(xpos + width/2, fem_absmax,  width, color=vc_fem,  label='FEM')
axes[1].set_yscale('log')
axes[1].set_xticks(xpos); axes[1].set_xticklabels([vlabels[v] for v in vn])
axes[1].set_ylabel('Max absolute $L^2$ error')
axes[1].set_title('Absolute $L^2$ (max over $t$)', fontsize=12, weight='bold')
axes[1].grid(True, alpha=0.3, axis='y', which='both'); axes[1].legend()

fig.suptitle('PINN vs FEM - Summary', fontsize=14, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'summary_bars.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'summary_bars.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## Cell 13 — Save raw data + LaTeX table, copy results back to Drive


In [ ]:
np.savez(os.path.join(OUT_DIR, 'errors.npz'),
         times=ta,
         **{f'pinn_rel_{v}': pinn_rel[v] for v in vn},
         **{f'pinn_abs_{v}': pinn_abs[v] for v in vn},
         **{f'fem_rel_{v}':  fem_rel[v]  for v in vn},
         **{f'fem_abs_{v}':  fem_abs[v]  for v in vn})

tex_path = os.path.join(OUT_DIR, 'results_table.tex')
with open(tex_path, 'w') as f:
    f.write(r"""\begin{table}[h]
\centering
\caption{PINN vs FEM: maximum $L^2$ errors on the """ + f"{nx}$\\times${nx}" +
            r""" evaluation grid over $t \in [0,1]$ (relative errors for $B,I,S$ from $t \ge 0.1$).}
\label{tab:pinn_vs_fem}
\begin{tabular}{lcccc}
\toprule
Variable & PINN rel.\ $L^2$ & FEM rel.\ $L^2$ & PINN abs.\ $L^2$ & FEM abs.\ $L^2$ \\
\midrule
""")
    for v, prv, frv, pav, fav in zip(vn, pinn_relmax, fem_relmax,
                                     pinn_absmax, fem_absmax):
        f.write(f"  ${v}$ & {prv:.2e} & {frv:.2e} & {pav:.2e} & {fav:.2e} \\\\\n")
    f.write(r"""\bottomrule
\end{tabular}
\end{table}
""")

# Copy results back to Google Drive (so you have them after the runtime ends)
DRIVE_OUT = '/content/drive/MyDrive/PINN_FEM/comparison_results'
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in os.listdir(OUT_DIR):
    shutil.copy(os.path.join(OUT_DIR, f), os.path.join(DRIVE_OUT, f))

print(f"\nAll artifacts saved to:")
print(f"  {OUT_DIR}/             (Colab runtime, lost on disconnect)")
print(f"  {DRIVE_OUT}/  (Google Drive, persistent)")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"    - {f}")


## Cell 14 — Figure 4: Exact vs PINN vs FEM solution snapshot

Plots the actual solution fields (not the errors) for all five variables on a 3×5 grid (Exact / PINN / FEM × T, B, O, I, S). Each column uses a single shared colour scale so the three methods are directly comparable for the eye.

Change `t_snap_solution` to plot a different time. This cell depends on variables defined in Cells 5–8 (the loaded model, the FEM directories, the grid). Run those first.


In [ ]:
# Solution snapshot — Exact vs PINN vs FEM at one time
t_snap_solution = 0.5     # change to 0.25, 0.75, 1.0, etc., to plot other times

# Find the closest FEM time
i_sol = int(np.argmin(np.abs(ta - t_snap_solution)))
t_sol_actual = ta[i_sol]
t_sol_name = time_names[i_sol]
print(f"Plotting Exact vs PINN vs FEM at t = {t_sol_actual:.2f} "
      f"(file '{t_sol_name}')")

# Build the three solutions on the grid
X = x_ref.reshape(nx, nx); Y = y_ref.reshape(nx, nx)

ex_field = {}
for v in vn:
    ex_field[v] = np.loadtxt(
        os.path.join(EX_DIR, ex_sub_map[v], t_sol_name))[:, 2].reshape(nx, nx)

pinn_pred_grid = predict_pinn(x_ref, y_ref, t_sol_actual)
pinn_field = {v: pinn_pred_grid[:, j].reshape(nx, nx)
              for j, v in enumerate(vn)}

fem_field = {}
for v in vn:
    fem_field[v] = np.loadtxt(
        os.path.join(AP_DIR, ap_sub_map[v], t_sol_name))[:, 2].reshape(nx, nx)

# Per-variable colourmap (visually distinguishes the variables)
cmaps = {'T': 'Reds', 'B': 'Blues', 'O': 'Greens',
         'I': 'Oranges', 'S': 'Purples'}

fig, axes = plt.subplots(3, 5, figsize=(22, 12))
fig.suptitle(f'Solution snapshot at $t = {t_sol_actual:.2f}$  '
             f'(grid {nx}$\\times${nx})',
             fontsize=16, weight='bold', y=1.00)

row_labels = ['Exact', 'PINN', 'FEM']
row_data = [ex_field, pinn_field, fem_field]

for j, v in enumerate(vn):
    # Shared color scale across the 3 panels for this variable
    vmin = min(ex_field[v].min(), pinn_field[v].min(), fem_field[v].min())
    vmax = max(ex_field[v].max(), pinn_field[v].max(), fem_field[v].max())
    for i in range(3):
        ax = axes[i, j]
        im = ax.pcolormesh(X, Y, row_data[i][v], cmap=cmaps[v],
                           shading='auto', vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_aspect('equal')
        ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1])
        if i == 0:
            ax.set_title(vlabels[v], fontsize=14, weight='bold')
        if j == 0:
            ax.set_ylabel(row_labels[i], fontsize=13, weight='bold',
                          rotation=90, labelpad=10)

plt.tight_layout()

# Save & copy to Drive
fname_pdf = f'solution_exact_pinn_fem_t{t_sol_actual:.2f}.pdf'
fname_png = f'solution_exact_pinn_fem_t{t_sol_actual:.2f}.png'
plt.savefig(os.path.join(OUT_DIR, fname_pdf), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, fname_png), dpi=150, bbox_inches='tight')
plt.show()

# Also copy to Drive so it persists after the runtime ends
shutil.copy(os.path.join(OUT_DIR, fname_pdf),
            os.path.join(DRIVE_OUT, fname_pdf))
shutil.copy(os.path.join(OUT_DIR, fname_png),
            os.path.join(DRIVE_OUT, fname_png))
print(f"Saved to {OUT_DIR}/{fname_pdf}")
print(f"Copied to {DRIVE_OUT}/{fname_pdf}")
